# 02_Feature_Extraction

## Purpose

This notebook extracts deep features from the preprocessed Navarasa dataset using the pretrained Vision Transformer (ViT) and Swin Transformer models. The extracted feature representations are saved for subsequent hybrid model training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

CLASS_NAMES = [
    "Adbhuta", "Bhayanaka", "Bibhatsya",
    "Hasya", "Karuna", "Raudra",
    "Shanta", "Shringara", "Veera"
]
NUM_CLASSES = len(CLASS_NAMES)


In [ ]:
BASE_DIR = "/content/drive/MyDrive/NAVARASA_PROJECT_DRIVE1"

DATASET_DIR = os.path.join(BASE_DIR, "split_dataset")
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "val")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

SWIN_PATH = os.path.join(BASE_DIR, "models/swin/swin_best.pth")
VIT_PATH  = os.path.join(BASE_DIR, "models/vit/vit_best.pth")


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

def get_loader(path, shuffle=False):
    return DataLoader(
        datasets.ImageFolder(root=path, transform=transform),
        batch_size=8,
        shuffle=shuffle,
        num_workers=0
    )

train_loader = get_loader(TRAIN_DIR, shuffle=True)
val_loader   = get_loader(VAL_DIR)
test_loader  = get_loader(TEST_DIR)


In [ ]:
def load_swin(path):
    m = timm.create_model(
        "swin_base_patch4_window7_224",
        pretrained=False,
        num_classes=NUM_CLASSES
    )
    m.load_state_dict(torch.load(path, map_location=device))
    m.head = nn.Identity()
    m.eval()
    return m.to(device)

def load_vit(path):
    m = timm.create_model(
        "vit_base_patch16_224",
        pretrained=False,
        num_classes=NUM_CLASSES
    )
    m.load_state_dict(torch.load(path, map_location=device))
    m.head = nn.Identity()
    m.eval()
    return m.to(device)

swin = load_swin(SWIN_PATH)
vit  = load_vit(VIT_PATH)

print("✅ Swin & ViT loaded as feature extractors")


In [ ]:
def extract_features(loader, split_name):
    swin_feats, vit_feats, labels = [], [], []

    with torch.no_grad():
        for i, (imgs, lbls) in enumerate(loader):
            imgs = imgs.to(device)

            s = swin(imgs)
            s = s.permute(0,3,1,2)
            s = F.adaptive_avg_pool2d(s,1).squeeze(-1).squeeze(-1)

            v = vit(imgs)

            swin_feats.append(s.cpu())
            vit_feats.append(v.cpu())
            labels.append(lbls)

            if i % 20 == 0:
                print(f"{split_name}: batch {i}/{len(loader)} done")

    return (
        torch.cat(swin_feats),
        torch.cat(vit_feats),
        torch.cat(labels)
    )


In [ ]:
FEATURE_DIR = os.path.join(BASE_DIR, "extracted_features")
os.makedirs(FEATURE_DIR, exist_ok=True)

X_swin_tr, X_vit_tr, y_tr = extract_features(train_loader, "TRAIN")
torch.save({"X_swin": X_swin_tr, "X_vit": X_vit_tr, "y": y_tr},
           os.path.join(FEATURE_DIR, "train.pt"))

X_swin_vl, X_vit_vl, y_vl = extract_features(val_loader, "VAL")
torch.save({"X_swin": X_swin_vl, "X_vit": X_vit_vl, "y": y_vl},
           os.path.join(FEATURE_DIR, "val.pt"))

X_swin_te, X_vit_te, y_te = extract_features(test_loader, "TEST")
torch.save({"X_swin": X_swin_te, "X_vit": X_vit_te, "y": y_te},
           os.path.join(FEATURE_DIR, "test.pt"))

print("Feature extraction completed successfully.")


In [ ]:
for split in ["train","val","test"]:
    data = torch.load(os.path.join(FEATURE_DIR, f"{split}.pt"))
    print(f"\n{split.upper()} SET")
    print("Swin features :", data["X_swin"].shape)
    print("ViT features  :", data["X_vit"].shape)
    print("Labels        :", data["y"].shape)
